# Part F — Deep Learning Using TensorFlow/Keras

Problems 19–20 use the same Fashion-MNIST task and the same small training, validation and test subsets as Part E. Run Part E first so its measured results are available for the comparison.

## Problem 19. Neural network using Keras

In [ ]:
import json
import time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
tf.keras.utils.set_random_seed(SEED)

(train_images, train_labels), (test_images, test_labels) = keras.datasets.fashion_mnist.load_data()
train_images = train_images.astype("float32") / 255.0
test_images = test_images.astype("float32") / 255.0

x_train = train_images[:10000]
y_train = train_labels[:10000]
x_val = train_images[10000:12000]
y_val = train_labels[10000:12000]
x_test = test_images[:2000]
y_test = test_labels[:2000]

class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]
print("Training samples:", len(x_train))
print("Validation samples:", len(x_val))
print("Test samples:", len(x_test))

model = keras.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(10)
])
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)
model.summary()

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=1, restore_best_weights=True
)
start_time = time.perf_counter()
history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=2,
    batch_size=128,
    callbacks=[early_stopping],
    verbose=1
)
training_time = time.perf_counter() - start_time

train_loss, train_accuracy = model.evaluate(x_train, y_train, verbose=0)
val_loss, val_accuracy = model.evaluate(x_val, y_val, verbose=0)
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
parameter_count = model.count_params()

keras_results = {
    "framework": "TensorFlow/Keras",
    "setting": "Adam learning rate 0.001",
    "learning_rate": 0.001,
    "epochs_completed": len(history.history["loss"]),
    "train_accuracy": float(train_accuracy),
    "validation_accuracy": float(val_accuracy),
    "test_accuracy": float(test_accuracy),
    "training_time_seconds": training_time,
    "test_loss": float(test_loss),
    "parameter_count": int(parameter_count),
    "train_samples": len(x_train),
    "validation_samples": len(x_val),
    "test_samples": len(x_test)
}
print("Keras results:")
for key, value in keras_results.items():
    print(f"{key}: {value}")

model.save("keras_model.keras")
Path("keras_results.json").write_text(json.dumps(keras_results, indent=2))
print("Saved keras_model.keras and keras_results.json")

## Training plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history.history["loss"], marker="o", label="Training loss")
axes[0].plot(history.history["val_loss"], marker="o", label="Validation loss")
axes[0].set_title("Keras loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[1].plot(history.history["accuracy"], marker="o", label="Training accuracy")
axes[1].plot(history.history["val_accuracy"], marker="o", label="Validation accuracy")
axes[1].set_title("Keras accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.tight_layout()
plt.savefig("keras_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## Saved model and prediction

In [ ]:
reloaded_model = keras.models.load_model("keras_model.keras")
probabilities = tf.nn.softmax(reloaded_model(x_test[:5], training=False), axis=1).numpy()
predictions = probabilities.argmax(axis=1)
for index, prediction in enumerate(predictions):
    print(f"Sample {index + 1}: predicted {class_names[int(prediction)]}; actual {class_names[int(y_test[index])]}")

## Problem 20. PyTorch versus TensorFlow/Keras

In [ ]:
pytorch_file = Path("pytorch_results.json")
if not pytorch_file.exists():
    raise FileNotFoundError("Run Part_E_PyTorch.ipynb first to create pytorch_results.json.")

pytorch_results = json.loads(pytorch_file.read_text())
comparison = __import__("pandas").DataFrame([
    {
        "Framework": "PyTorch",
        "Architecture": "Flatten → Dense(128, ReLU) → Dense(10)",
        "Parameters": pytorch_results["parameter_count"],
        "Training accuracy": pytorch_results["train_accuracy"],
        "Validation accuracy": pytorch_results["validation_accuracy"],
        "Test accuracy": pytorch_results["test_accuracy"],
        "Training time (s)": pytorch_results["training_time_seconds"],
        "Ease of implementation/debugging": "Explicit training loop; flexible step-by-step debugging"
    },
    {
        "Framework": "TensorFlow/Keras",
        "Architecture": "Flatten → Dense(128, ReLU) → Dense(10)",
        "Parameters": keras_results["parameter_count"],
        "Training accuracy": keras_results["train_accuracy"],
        "Validation accuracy": keras_results["validation_accuracy"],
        "Test accuracy": keras_results["test_accuracy"],
        "Training time (s)": keras_results["training_time_seconds"],
        "Ease of implementation/debugging": "Concise model.fit API; callbacks and evaluation built in"
    }
])
display(comparison.round(4))
comparison.to_csv("framework_comparison.csv", index=False)

print("Both models use the same task, architecture, subset sizes and batch size, but framework overhead, shuffling and hardware can affect runtime.")
print("Compare measured test and validation results rather than assuming either framework is always better.")
print("Parameter counts should match because the dense-layer architecture is the same.")